# NeuroScan — VGG16
Two-phase transfer learning on Brain Tumor MRI dataset (4 classes).
Phase 1: classifier head only (10 epochs, AdamW).
Phase 2: features[24:] (last conv block) + classifier (15 epochs, AdamW + CosineAnnealingLR).
Set accelerator to **GPU T4 x2** before running.

In [ ]:
import subprocess, torch

r = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU       :', r.stdout.strip() or 'none')
print('PyTorch   :', torch.__version__)
print('CUDA      :', torch.cuda.is_available())

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

if device.type == 'cuda':
    try:
        torch.zeros(1, device=device)
        print(f'GPU OK    : {torch.cuda.get_device_name(0)}')
    except Exception as e:
        print(f'GPU fail  : {e} — falling back to CPU')
        device, use_amp = torch.device('cpu'), False

print(f'\nDevice: {device}  |  AMP: {use_amp}')

In [ ]:
import os

def find_dataset_root(base='/kaggle/input'):
    for root, dirs, _ in os.walk(base):
        if 'Training' in dirs and 'Testing' in dirs:
            return root
    return None

DATA_DIR = find_dataset_root()
if DATA_DIR is None:
    raise FileNotFoundError('Dataset not found — attach masoudnickparvar/brain-tumor-mri-dataset')

print(f'Dataset  : {DATA_DIR}')
print(f'Training : {os.path.isdir(DATA_DIR + "/Training")}')
print(f'Testing  : {os.path.isdir(DATA_DIR + "/Testing")}')

In [ ]:
import copy, time, json
import numpy as np
import torch.nn as nn
from torch import optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report)
from tqdm import tqdm

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = 4
IMG_SIZE    = 224
BATCH_SIZE  = 32 if device.type == 'cuda' else 16
EPOCHS_P1   = 10
EPOCHS_P2   = 15
LR_P1       = 1e-3
LR_P2       = 5e-5
NUM_WORKERS = 2
OUTPUT_DIR  = '/kaggle/working/neuroscan/vgg16'
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]

print(f'Batch size : {BATCH_SIZE}')
print(f'Epochs     : {EPOCHS_P1} + {EPOCHS_P2}')
print(f'Output dir : {OUTPUT_DIR}')

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples   = []
        self.label_map = {c: i for i, c in enumerate(CLASS_NAMES)}
        self.transform = transform
        for cls in CLASS_NAMES:
            d = os.path.join(root, cls)
            if not os.path.isdir(d): continue
            for f in os.listdir(d):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(d, f), self.label_map[cls]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, lbl = self.samples[i]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, lbl


aug = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
basic = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_set = BrainTumorDataset(os.path.join(DATA_DIR, 'Training'), aug)
test_set  = BrainTumorDataset(os.path.join(DATA_DIR, 'Testing'),  basic)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))

print(f'Train: {len(train_set)}  |  Test: {len(test_set)}')

In [ ]:
# Build VGG16 with custom head
# Classifier structure mirrors the backend so weights are directly loadable.
# Original VGG16 classifier: [Linear(25088,4096), ReLU, Drop, Linear(4096,4096), ReLU, Drop, Linear(4096,1000)]
# We keep everything except the final Linear and add our own Dropout + Linear(4096, 4).
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(
    *list(model.classifier.children())[:-1],   # drop Linear(4096, 1000)
    nn.Dropout(p=0.4),
    nn.Linear(4096, NUM_CLASSES),
)

# Phase 1: freeze all feature layers
for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)

t = sum(p.numel() for p in model.parameters() if p.requires_grad)
n = sum(p.numel() for p in model.parameters())
print('VGG16 loaded')
print(f'Phase 1 trainable: {t:,} / {n:,} ({100*t/n:.1f}%)')

In [ ]:
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    loss_sum = correct = total = 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    if scheduler: scheduler.step()
    return loss_sum / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / total, correct / total


def run_phase(phase_name, epochs, lr, unfreeze_fn=None):
    if unfreeze_fn: unfreeze_fn()
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n = sum(p.numel() for p in model.parameters())
    print(f'\n-- {phase_name} | trainable {t:,}/{n:,} ({100*t/n:.1f}%) | lr={lr}')

    ckpt_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    history   = []

    best_acc, best_state = 0.0, copy.deepcopy(model.state_dict())
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler)
        va_loss, va_acc = eval_epoch(model, test_loader)
        history.append({'epoch': epoch, 'train_loss': round(tr_loss, 6),
                        'train_acc': round(tr_acc, 6), 'val_loss': round(va_loss, 6),
                        'val_acc': round(va_acc, 6)})
        print(f'  Epoch {epoch:02d}/{epochs}  '
              f'train={tr_loss:.4f}/{tr_acc:.4f}  '
              f'val={va_loss:.4f}/{va_acc:.4f}  '
              f'({time.time()-t0:.1f}s)')
        if va_acc > best_acc:
            best_acc   = va_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, os.path.join(ckpt_dir, f'{phase_name}_best.pt'))
        if epoch % 5 == 0:
            torch.save(model.state_dict(),
                       os.path.join(ckpt_dir, f'{phase_name}_epoch{epoch:02d}.pt'))

    model.load_state_dict(best_state)
    with open(os.path.join(ckpt_dir, f'{phase_name}_history.json'), 'w') as fh:
        json.dump(history, fh, indent=2)
    print(f'  Best val acc ({phase_name}): {best_acc:.4f}')
    return best_acc

In [ ]:
# Phase 1 — classifier only
run_phase('phase1', EPOCHS_P1, LR_P1)

In [ ]:
# Phase 2 — unfreeze features[24:] (last conv block: 3x Conv512) + classifier
# VGG16 features layout: [0-4]=block1, [5-9]=block2, [10-16]=block3,
#                         [17-23]=block4, [24-30]=block5 (last)
def unfreeze_p2():
    for param in model.parameters(): param.requires_grad = False
    for i, layer in enumerate(model.features):
        if i >= 24:
            for param in layer.parameters(): param.requires_grad = True
    for param in model.classifier.parameters(): param.requires_grad = True

run_phase('phase2', EPOCHS_P2, LR_P2, unfreeze_fn=unfreeze_p2)

In [ ]:
# Final evaluation on full test set (400 x 4 = 1600 samples)
@torch.no_grad()
def full_eval(model, loader):
    model.eval()
    preds, labels = [], []
    for imgs, lbls in loader:
        preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y, p = np.array(labels), np.array(preds)
    print(f'Accuracy  : {accuracy_score(y, p):.4f}')
    print(f'F1        : {f1_score(y, p, average="weighted"):.4f}')
    print(f'Precision : {precision_score(y, p, average="weighted", zero_division=0):.4f}')
    print(f'Recall    : {recall_score(y, p, average="weighted"):.4f}')
    print()
    print(classification_report(y, p, target_names=CLASS_NAMES))
    return y, p

y_true, y_pred = full_eval(model, test_loader)

out_dir = os.path.join(OUTPUT_DIR, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'vgg16_neuroscan.pt')
torch.save(model.state_dict(), out_path)
print(f'\nWeights saved: {out_path}')

In [ ]:
print('Output files:')
for root, dirs, files in os.walk('/kaggle/working/neuroscan'):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')